# CineData Analytics: Agente Text-to-SQL

Notebook do agente que responde perguntas em linguagem natural sobre o catálogo de filmes da CineData.

**Seção 1:** conexão com o banco e exploração dos dados (sem IA, sem gastar requisições).


## 1. Setup


In [1]:
import sqlite3
from contextlib import closing
from pathlib import Path

import pandas as pd

# Mostra números com separador de milhar e 2 casas, em vez de notação científica (1.2e+10)
pd.set_option("display.float_format", "{:,.2f}".format)

# O notebook roda de dentro da pasta notebooks/, então o banco fica um nível acima
CAMINHO_BANCO = Path("..") / "data" / "cinerocket.db"

if not CAMINHO_BANCO.exists():
    raise FileNotFoundError(f"Banco não encontrado em {CAMINHO_BANCO.resolve()}. Veja o README.")


## 2. Conexão somente leitura


In [2]:
# Somente leitura como foi especificado no documento da atividade

def conectar() -> sqlite3.Connection:
    # Abre o banco em modo de somente leitura (o próprio SQLite recusa qualquer escrita)
    uri = CAMINHO_BANCO.resolve().as_uri() + "?mode=ro"
    return sqlite3.connect(uri, uri=True)


def consultar(sql: str) -> pd.DataFrame:
    # Faz uma consulta SQL e retorna o resultado em forma de tabela (DataFrame)
    with closing(conectar()) as conn:
        return pd.read_sql_query(sql, conn)


In [3]:
# Teste de segurança
# Tentativa de apagar dados (tem que falhar)

try:
    consultar("DELETE FROM dim_genres")
    print("PERIGO: a escrita foi permitida!")
except Exception as erro:
    print(f"Escrita bloqueada, como esperado: {erro}")


Escrita bloqueada, como esperado: Execution failed on sql 'DELETE FROM dim_genres': attempt to write a readonly database


## 3. Explorando o banco


In [4]:
# Consulta as tabelas do banco e o número de linhas em cada uma (ter uma visão geral do conteúdo)

tabelas = consultar("""
    SELECT name AS tabela
    FROM sqlite_master
    WHERE type = 'table' AND name != 'alembic_version'
    ORDER BY name
""")
tabelas["linhas"] = [consultar(f"SELECT COUNT(*) AS n FROM {t}")["n"][0] for t in tabelas["tabela"]]
tabelas


,tabela,linhas
0,bridge_movie_company,116326
1,bridge_movie_genre,121521
2,bridge_movie_person,745450
3,dim_companies,45941
4,dim_genres,19
5,dim_movies,95645
6,dim_people,424656
7,dim_reviews,40267
8,fact_movies_performance,95645
9,movie_reviews,43666


In [5]:
# PRAGMA table_info é um comando do SQLite que descreve as colunas de uma tabela

def colunas(tabela: str) -> pd.DataFrame:
    # Mostra as colunas de uma tabela: nome, tipo e se aceita nulo
    info = consultar(f"PRAGMA table_info({tabela})")
    return info[["name", "type", "notnull"]].rename(
        columns={"name": "coluna", "type": "tipo", "notnull": "obrigatoria"}
    )


for tabela in tabelas["tabela"]:
    print(f"\n=== {tabela} ===")
    display(colunas(tabela))

# 1 significa que o campo nunca é null



=== bridge_movie_company ===


,coluna,tipo,obrigatoria
0,sk_movie_id,VARCHAR(64),1
1,sk_company_id,VARCHAR(64),1



=== bridge_movie_genre ===


,coluna,tipo,obrigatoria
0,sk_movie_id,VARCHAR(64),1
1,sk_genre_id,VARCHAR(64),1



=== bridge_movie_person ===


...

,coluna,tipo,obrigatoria
0,sk_review_id,VARCHAR(64),1
1,sk_movie_id,VARCHAR(64),1
2,qtd_avaliacoes_usuarios,INTEGER,1
3,nota_media_usuarios,DOUBLE,0



=== fact_movies_performance ===


,coluna,tipo,obrigatoria
0,sk_movie_id,VARCHAR(64),1
1,orcamento_usd,"NUMERIC(18, 2)",0
2,receita_usd,"NUMERIC(18, 2)",0
3,lucro_usd,"NUMERIC(18, 2)",1
4,orcamento_brl,"NUMERIC(18, 2)",0
5,receita_brl,"NUMERIC(18, 2)",0
6,lucro_brl,"NUMERIC(18, 2)",1
7,popularidade,DOUBLE,0
8,nota_tmdb,DOUBLE,0
9,qtd_tmdb,INTEGER,0



=== movie_reviews ===


,coluna,tipo,obrigatoria
0,id,INTEGER,1
1,sk_movie_review_id,VARCHAR(64),1
2,sk_movie_id,VARCHAR(64),1
3,name,VARCHAR(120),1
4,rating,DOUBLE,1
5,text,VARCHAR(4000),1
6,created_at,DATETIME,1


In [6]:
# Amostra de 5 filmes juntando as duas tabelas pelo sk_movie_id

consultar("""
    SELECT m.titulo, m.ano_lancamento, f.receita_brl, f.nota_imdb, f.popularidade
    FROM dim_movies m
    JOIN fact_movies_performance f ON f.sk_movie_id = m.sk_movie_id
    LIMIT 5
""")


,titulo,ano_lancamento,receita_brl,nota_imdb,popularidade
0,Rings,2017,"261,480,485.10",4.50,24.58
1,Mixtape,2021,NaN,6.60,8.93
2,Grizzly Ii: Revenge,2020,NaN,NaN,NaN
3,Billy Joel - Live At Yankee Stadium,2022,NaN,7.80,2.98
4,Bad Boys For Life,2020,"1,774,944,223.43",6.50,46.62


## 4. Qualidade dos dados


In [7]:
# Quantos filmes têm receita e orçamento informados?

consultar("""
    SELECT
        COUNT(*)                                                    AS total_filmes,
        SUM(receita_usd IS NOT NULL)                                AS com_receita,
        SUM(orcamento_usd IS NOT NULL)                              AS com_orcamento,
        SUM(receita_usd IS NOT NULL AND orcamento_usd IS NOT NULL)  AS com_ambos
    FROM fact_movies_performance
""")


,total_filmes,com_receita,com_orcamento,com_ambos
0,95645,3373,7926,1630


In [8]:
# Como o lucro foi calculado quando falta receita ou orçamento?

consultar("""
    SELECT
        CASE
            WHEN receita_usd IS NULL AND orcamento_usd IS NULL THEN 'sem receita e sem orçamento'
            WHEN receita_usd IS NULL                          THEN 'só orçamento'
            WHEN orcamento_usd IS NULL                        THEN 'só receita'
            ELSE 'receita e orçamento'
        END                                  AS situacao,
        COUNT(*)                             AS filmes,
        SUM(lucro_usd = 0)                   AS lucro_zero,
        SUM(lucro_usd = -orcamento_usd)      AS lucro_igual_menos_orcamento,
        SUM(lucro_usd = receita_usd)         AS lucro_igual_receita
    FROM fact_movies_performance
    GROUP BY situacao
""")


,situacao,filmes,lucro_zero,lucro_igual_menos_orcamento,lucro_igual_receita
0,receita e orçamento,1630,56,0.00,0.00
1,sem receita e sem orçamento,85976,85976,NaN,NaN
2,só orçamento,6296,0,"6,296.00",NaN
3,só receita,1743,0,NaN,"1,743.00"


In [9]:
# Notas: valores nulos e zeros
consultar("""
    SELECT
        SUM(nota_imdb IS NULL)  AS imdb_nula,
        SUM(nota_tmdb = 0)      AS tmdb_zero,
        SUM(qtd_tmdb = 0)       AS tmdb_sem_votos
    FROM fact_movies_performance
""")


,imdb_nula,tmdb_zero,tmdb_sem_votos
0,12674,36185,33207


In [10]:
# Distribuição dos anos de lançamento
consultar("""
    SELECT ano_lancamento, COUNT(*) AS filmes
    FROM dim_movies
    GROUP BY ano_lancamento
    ORDER BY ano_lancamento
""")


,ano_lancamento,filmes
0,2016,12051
1,2017,12987
2,2018,13013
3,2019,13349
4,2020,10953
5,2021,11052
6,2022,11363
7,2023,8993
8,2024,1875
9,2025,5


In [11]:
# Pessoas por categoria (tipo_pessoa)  e colunas sem dados

display(consultar("SELECT tipo_pessoa, COUNT(*) AS pessoas FROM dim_people GROUP BY tipo_pessoa"))
display(consultar("SELECT COUNT(idioma_original) AS filmes_com_idioma FROM dim_movies"))


,tipo_pessoa,pessoas
0,Ator,273400
1,Diretor,65200
2,Roteirista,86056


,filmes_com_idioma
0,0


### Pontos curiosos

Estas regras vão entrar no prompt do agente, senão ele erra os cálculos:

- **Receita é rara:** só ~3,5% dos filmes têm receita informada e só ~1.600 têm receita **e** orçamento.
- **O lucro engana:** a coluna `lucro` nunca é nula. Sem receita, ela vale `-orçamento`; sem orçamento, vale a própria receita; sem os dois, vale 0. Lucro e margem só fazem sentido quando **receita e orçamento** estão informados.
- **Nota TMDB = 0** significa "sem votos", não nota zero. Muitas notas IMDb são nulas.
- **Anos:** o catálogo vai de 2016 a 2029, concentrado até 2024.
- **Pessoas:** `bridge_movie_person` não diz o papel; ele está em `dim_people.tipo_pessoa` (Ator, Diretor, Roteirista). A mesma pessoa pode aparecer com tipos diferentes.
- **Gêneros** estão em inglês (Action, Drama...). `idioma_original` está vazio.


## 5. Respostas de referência

Algumas perguntas do desafio resolvidas à mão. Servem para comparar com as respostas do agente depois.


In [12]:
# Top 10 filmes com maior receita em R$

consultar("""
    SELECT m.titulo, m.ano_lancamento, f.receita_brl
    FROM fact_movies_performance f
    JOIN dim_movies m ON m.sk_movie_id = f.sk_movie_id
    WHERE f.receita_brl IS NOT NULL
    ORDER BY f.receita_brl DESC
    LIMIT 10
""")


,titulo,ano_lancamento,receita_brl
0,Avatar: The Way Of Water,2022,"12,390,136,500.54"
1,Avengers: Endgame,2019,"11,094,720,000.00"
2,Spider-man: No Way Home,2021,"10,977,782,882.74"
3,Avengers: Infinity War,2018,"7,190,430,847.63"
4,Top Gun: Maverick,2022,"7,160,804,869.01"
5,Barbie,2023,"6,856,159,007.38"
6,The Super Mario Bros. Movie,2023,"6,838,413,799.10"
7,The Lion King,2019,"6,227,552,146.58"
8,Frozen Ii,2019,"6,094,028,191.32"
9,Jurassic World: Fallen Kingdom,2018,"4,934,822,930.85"


In [13]:
# Lucro médio por gênero, considerando apenas filmes com receita e orçamento informados

consultar("""
    SELECT g.nome_genero, COUNT(*) AS filmes, ROUND(AVG(f.lucro_brl), 2) AS lucro_medio_brl
    FROM fact_movies_performance f
    JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
    JOIN dim_genres g          ON g.sk_genre_id  = bg.sk_genre_id
    WHERE f.receita_brl IS NOT NULL AND f.orcamento_brl IS NOT NULL
    GROUP BY g.nome_genero
    ORDER BY lucro_medio_brl DESC
""")


,nome_genero,filmes,lucro_medio_brl
0,Science Fiction,134,"755,742,404.17"
1,Adventure,246,"724,425,244.47"
2,Animation,97,"509,966,381.98"
3,Fantasy,134,"509,886,798.93"
4,Family,142,"460,045,371.81"
5,Action,379,"457,268,303.09"
6,War,57,"247,003,988.77"
7,Comedy,403,"202,550,286.87"
8,Music,45,"187,931,397.14"
9,Horror,168,"166,814,396.19"


In [14]:
# Diretores com maior nota média IMDb (mínimo de 5 filmes com nota)

consultar("""
    SELECT p.nome_pessoa AS diretor, COUNT(*) AS filmes, ROUND(AVG(f.nota_imdb), 2) AS nota_media_imdb
    FROM dim_people p
    JOIN bridge_movie_person bp     ON bp.sk_person_id = p.sk_person_id
    JOIN fact_movies_performance f  ON f.sk_movie_id   = bp.sk_movie_id
    WHERE p.tipo_pessoa = 'Diretor' AND f.nota_imdb IS NOT NULL
    GROUP BY p.sk_person_id
    HAVING COUNT(*) >= 5
    ORDER BY nota_media_imdb DESC
    LIMIT 10
""")


,diretor,filmes,nota_media_imdb
0,Scott Wozniak,5,9.34
1,Yūichirō Hayashi,8,9.19
2,Jun Shishido,8,9.19
3,Trevor L. Allen,6,9.15
4,Alonso O. Lara,14,9.09
5,Tokio Igarashi,5,9.00
6,Erlik,6,8.95
7,Stuart Webster,5,8.88
8,Mark Fischbach,6,8.83
9,John D. Boswell,8,8.70


In [15]:
# Dupla ator–diretor que mais trabalhou junta

consultar("""
    SELECT a.nome_pessoa AS ator, d.nome_pessoa AS diretor, COUNT(*) AS filmes_juntos
    FROM bridge_movie_person ba
    JOIN dim_people a          ON a.sk_person_id = ba.sk_person_id AND a.tipo_pessoa = 'Ator'
    JOIN bridge_movie_person bd ON bd.sk_movie_id = ba.sk_movie_id
    JOIN dim_people d          ON d.sk_person_id = bd.sk_person_id AND d.tipo_pessoa = 'Diretor'
    WHERE a.nome_pessoa != d.nome_pessoa
    GROUP BY a.sk_person_id, d.sk_person_id
    ORDER BY filmes_juntos DESC
    LIMIT 10
""")


,ator,diretor,filmes_juntos
0,Joe Anoa'i,Kevin Dunn,37
1,Colby Lopez,Kevin Dunn,32
2,David Love,Chad Payne,31
3,Anton Pelizzari,Chad Payne,31
4,Cameron Nichols,Chad Payne,31
5,Jazzyjoeyjr,Chad Payne,31
6,Ahomas Hailwuttem,Chad Payne,31
7,John Whinfield,Chad Payne,31
8,Vivica A. Fox,David Decoteau,25
9,Jeff Kirkendall,Mark Polonia,25


## 6. Configuração do agente


In [16]:
import os

import httpx
from dotenv import load_dotenv

# Lê o arquivo .env e coloca a chave nas variáveis de ambiente
load_dotenv()

if not os.getenv("OPENROUTER_API_KEY"):
    raise RuntimeError("Chave OPENROUTER_API_KEY não encontrada. Copie o .env.example para .env e preencha.")

# Modelo gratuito com suporte a ferramentas (tool calling)
MODELO = "openrouter:nvidia/nemotron-3.5-lightning:free"


In [ ]:
# raise_for_status() - se o OpenRouter responder com erro (por exemplo, 401 por chave inválida)
# httpx.get() - acessa o endereço do OpenRouter

def verificar_cota() -> None:
    # Consulta quantas requisições gratuitas ainda restam hoje (essa consulta não conta no limite)
    resposta = httpx.get(
        "https://openrouter.ai/api/v1/key",
        headers={"Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}"},
        timeout=20,
    )
    resposta.raise_for_status()
    cota = resposta.json()["data"]["free_model_daily_requests"]
    print(f"Requisições usadas hoje: {cota['used']}/{cota['limit']} (restam {cota['remaining']})")


verificar_cota()


Requisições usadas hoje: 0/50 (restam 50)


## 7. Ferramenta de execução de SQL

É a única ferramenta do agente: recebe o SQL gerado pelo modelo, executa no banco e devolve o resultado em texto.

**Proteções:**
- Só aceita consultas de leitura (`SELECT` ou `WITH`), além do `mode=ro` da conexão
- Devolve no máximo 50 linhas, para não estourar o contexto do modelo
- Se o SQL der erro, devolve a mensagem para o modelo corrigir e tentar de novo (`ModelRetry`)

**Por que CSV?** O resultado volta para o modelo como texto, e cada token conta. Medindo uma consulta de 50 filmes (título, ano, receita e nota) com o `tiktoken`:

| Formato | Tokens |
|---|---|
| CSV | 964 |
| Tabela do pandas | 1.176 (+22%) |
| Markdown | 1.454 (+51%) |
| JSON | 1.902 (+97%) |


In [20]:
from pydantic_ai import ModelRetry

LIMITE_LINHAS = 50


def executar_sql(sql: str) -> str:
    # Tira espaços e o ";" do final, que os modelos costumam colocar
    sql = sql.strip().rstrip(";").strip()

    # Só aceita consultas de leitura
    primeira_palavra = sql.split(maxsplit=1)[0].upper() if sql else ""
    if primeira_palavra not in ("SELECT", "WITH"):
        raise ModelRetry("Só posso executar consultas de leitura (SELECT ou WITH).")

    try:
        resultado = consultar(sql)
    except Exception as erro:
        raise ModelRetry(f"O SQL deu erro: {erro}\nCorrija a consulta e tente de novo.") from erro

    if resultado.empty:
        return "A consulta não retornou nenhuma linha."

    texto = resultado.head(LIMITE_LINHAS).to_csv(index=False)
    if len(resultado) > LIMITE_LINHAS:
        texto += f"\n(mostrando {LIMITE_LINHAS} de {len(resultado)} linhas)"
    return texto


### Testes da ferramenta (sem IA)


In [ ]:
# Consulta normal (com ";" no final, como o modelo costuma mandar)

print(executar_sql("SELECT nome_genero FROM dim_genres ORDER BY nome_genero LIMIT 5;"))

nome_genero
Action
Adventure
Animation
Comedy
Crime



In [ ]:
# Consulta que traz muitas linhas: tem que cortar em 50 (mostra só a última linha do texto)

print(executar_sql("SELECT titulo FROM dim_movies").splitlines()[-1])

(mostrando 50 de 95645 linhas)


In [ ]:
# Consulta sem resultado

print(executar_sql("SELECT * FROM dim_genres WHERE nome_genero = 'Novela'"))

A consulta não retornou nenhuma linha.


In [24]:
# Casos que precisam ser recusados: escrita, dois comandos juntos e coluna inexistente
casos = [
    "DELETE FROM dim_genres",
    "SELECT 1; DROP TABLE dim_genres",
    "SELECT receita FROM fact_movies_performance",
]

for sql in casos:
    try:
        executar_sql(sql)
        print(f"PERIGO: {sql!r} foi aceito!")
    except ModelRetry as erro:
        print(f"{sql!r}\n  -> {erro}\n")

'DELETE FROM dim_genres'
  -> Só posso executar consultas de leitura (SELECT ou WITH).

'SELECT 1; DROP TABLE dim_genres'
  -> O SQL deu erro: Execution failed on sql 'SELECT 1; DROP TABLE dim_genres': You can only execute one statement at a time.
Corrija a consulta e tente de novo.

'SELECT receita FROM fact_movies_performance'
  -> O SQL deu erro: Execution failed on sql 'SELECT receita FROM fact_movies_performance': no such column: receita
Corrija a consulta e tente de novo.



## 8. Prompt de sistema

É no prompt que a ferramenta `executar_sql` ensina pro agente como o banco funciona.

### 8.1 Schema do banco

O schema é gerado automaticamente a partir do próprio banco, para não ter erro de digitação nem coluna esquecida. Além das colunas e ligações entre tabelas, inclui os valores possíveis das colunas de categoria (gêneros, tipos de pessoa e status), para o modelo saber, por exemplo, que os gêneros estão em inglês.


In [25]:
# Colunas com poucos valores diferentes, que vale a pena listar para o modelo
COLUNAS_CATEGORICAS = [
    ("dim_genres", "nome_genero"),
    ("dim_people", "tipo_pessoa"),
    ("dim_movies", "status_filme"),
]


def gerar_schema() -> str:
    linhas = []
    for tabela in tabelas["tabela"]:
        colunas_tabela = consultar(f"PRAGMA table_info({tabela})")
        descricao = ", ".join(f"{col['name']} {col['type']}" for _, col in colunas_tabela.iterrows())
        linhas.append(f"- {tabela}({descricao})")

        # Ligações com outras tabelas (chaves estrangeiras)
        for _, fk in consultar(f"PRAGMA foreign_key_list({tabela})").iterrows():
            linhas.append(f"    {fk['from']} -> {fk['table']}.{fk['to']}")

    linhas.append("\nValores possíveis:")
    for tabela, coluna in COLUNAS_CATEGORICAS:
        valores = consultar(f"SELECT DISTINCT {coluna} FROM {tabela} WHERE {coluna} IS NOT NULL ORDER BY 1")
        linhas.append(f"- {tabela}.{coluna}: {', '.join(valores[coluna])}")

    return "\n".join(linhas)


SCHEMA = gerar_schema()
print(SCHEMA)


- bridge_movie_company(sk_movie_id VARCHAR(64), sk_company_id VARCHAR(64))
    sk_company_id -> dim_companies.sk_company_id
    sk_movie_id -> dim_movies.sk_movie_id
- bridge_movie_genre(sk_movie_id VARCHAR(64), sk_genre_id VARCHAR(64))
    sk_genre_id -> dim_genres.sk_genre_id
    sk_movie_id -> dim_movies.sk_movie_id
- bridge_movie_person(sk_movie_id VARCHAR(64), sk_person_id VARCHAR(64))
    sk_person_id -> dim_people.sk_person_id
    sk_movie_id -> dim_movies.sk_movie_id
- dim_companies(sk_company_id VARCHAR(64), nome_produtora VARCHAR(255))
- dim_genres(sk_genre_id VARCHAR(64), nome_genero VARCHAR(50))
- dim_movies(sk_movie_id VARCHAR(64), id_filme VARCHAR(50), titulo VARCHAR(500), data_lancamento DATE, ano_lancamento INTEGER, duracao_minutos INTEGER, idioma_original VARCHAR(10), status_filme VARCHAR(50), sinopse VARCHAR(4000), url_poster VARCHAR(2048), url_backdrop VARCHAR(2048))
- dim_people(sk_person_id VARCHAR(64), nome_pessoa VARCHAR(255), tipo_pessoa VARCHAR(20))
- dim_revie

### 8.2 Regras e exemplos

Decisões de negócio usadas no prompt:

- **Moeda padrão é o real (R$):** o público é brasileiro e as perguntas do desafio usam R$. Dólar só quando o usuário pedir.
- **Margem de lucro = lucro ÷ receita** ("de tudo que faturou, quanto virou lucro"). **Retorno (ROI) = lucro ÷ orçamento** ("quanto voltou para cada real investido").
- **Orçamento mínimo de US$ 10 mil em margem e retorno:** existem filmes com orçamento de US$ 50 ou US$ 128, claramente erros de cadastro, que dominariam qualquer ranking com margens de 100% e retornos de milhões de %. O corte em US$ 10 mil remove esses absurdos sem descartar produções de baixo orçamento reais. O agente avisa o usuário quando aplica esse filtro.
- **Exemplos (few-shot):** dois exemplos de pergunta e SQL ensinam os JOINs mais difíceis (gênero e papel da pessoa). Propositalmente, nenhum deles é uma pergunta do desafio, para não "entregar a resposta" ao modelo.


In [26]:
REGRAS_NEGOCIO = """
1. "Receita", "faturamento" e "bilheteria" são a mesma coisa.
2. Moeda: use as colunas em reais (_brl) por padrão. Use dólar (_usd) só se o usuário pedir.
3. Lucro: a coluna de lucro nunca é nula e engana: sem receita ela vale menos o orçamento. Em qualquer cálculo de lucro, margem ou retorno, use só filmes com receita E orçamento informados (IS NOT NULL).
4. Margem de lucro = lucro / receita. Retorno (ROI) = lucro / orçamento. Nesses dois cálculos, ignore filmes com orcamento_usd < 10000 (erros de cadastro) e avise o usuário desse filtro.
5. Notas: nota_tmdb = 0 significa "sem votos", então ignore esses casos. nota_imdb pode ser nula.
6. Pessoas: o papel (Ator, Diretor, Roteirista) está em dim_people.tipo_pessoa. A mesma pessoa pode aparecer mais de uma vez, uma por papel.
7. Gêneros estão em inglês: traduza a pergunta (ex.: "terror" -> 'Horror').
8. Nomes e títulos: busque com LIKE '%...%', porque o usuário pode não digitar o nome exato.
9. Avaliações dos usuários: estão em dim_reviews (qtd_avaliacoes_usuarios e nota_media_usuarios, de 0 a 10).
10. Datas: "últimos N anos" conta a partir do ano atual. O catálogo vai de 2016 a 2029, mas quase todos os filmes são até 2024. Avise se o período pedido tiver poucos dados.
"""

COMPORTAMENTO = """
- Sempre consulte o banco com a ferramenta executar_sql antes de responder. Nunca invente números.
- Em rankings sem quantidade definida, mostre os 10 primeiros.
- Responda em português, de forma curta e clara, com valores formatados (ex.: "R$ 12,4 bilhões").
- Diga quais filtros usou (ex.: "considerando só filmes com receita e orçamento informados").
- Se a pergunta não for sobre o catálogo de filmes, recuse educadamente.
"""

EXEMPLOS = """
Pergunta: Quais os 5 filmes de terror mais bem avaliados no IMDb, com pelo menos 1.000 votos?
SQL:
SELECT m.titulo, m.ano_lancamento, f.nota_imdb, f.qtd_imdb
FROM fact_movies_performance f
JOIN dim_movies m          ON m.sk_movie_id = f.sk_movie_id
JOIN bridge_movie_genre bg ON bg.sk_movie_id = f.sk_movie_id
JOIN dim_genres g          ON g.sk_genre_id = bg.sk_genre_id
WHERE g.nome_genero = 'Horror' AND f.qtd_imdb >= 1000
ORDER BY f.nota_imdb DESC
LIMIT 5

Pergunta: Em quantos filmes o Tom Hanks atuou e qual a nota média IMDb deles?
SQL:
SELECT p.nome_pessoa AS ator, COUNT(*) AS filmes, ROUND(AVG(f.nota_imdb), 2) AS nota_media_imdb
FROM dim_people p
JOIN bridge_movie_person bp    ON bp.sk_person_id = p.sk_person_id
JOIN fact_movies_performance f ON f.sk_movie_id = bp.sk_movie_id
WHERE p.tipo_pessoa = 'Ator' AND p.nome_pessoa LIKE '%Tom Hanks%'
GROUP BY p.sk_person_id
"""


In [27]:
from datetime import date

INSTRUCOES = f"""Você é o assistente de dados da CineData Analytics. Você responde perguntas sobre o catálogo de filmes para pessoas que não sabem SQL, consultando um banco SQLite com a ferramenta executar_sql. Hoje é {date.today():%d/%m/%Y}.

## Schema do banco
{SCHEMA}

## Regras de negócio
{REGRAS_NEGOCIO}
## Comportamento
{COMPORTAMENTO}
## Exemplos
{EXEMPLOS}"""

print(INSTRUCOES)


Você é o assistente de dados da CineData Analytics. Você responde perguntas sobre o catálogo de filmes para pessoas que não sabem SQL, consultando um banco SQLite com a ferramenta executar_sql. Hoje é 02/10/2026.

## Schema do banco
- bridge_movie_company(sk_movie_id VARCHAR(64), sk_company_id VARCHAR(64))
    sk_company_id -> dim_companies.sk_company_id
    sk_movie_id -> dim_movies.sk_movie_id
- bridge_movie_genre(sk_movie_id VARCHAR(64), sk_genre_id VARCHAR(64))
    sk_genre_id -> dim_genres.sk_genre_id
    sk_movie_id -> dim_movies.sk_movie_id
- bridge_movie_person(sk_movie_id VARCHAR(64), sk_person_id VARCHAR(64))
    sk_person_id -> dim_people.sk_person_id
    sk_movie_id -> dim_movies.sk_movie_id
- dim_companies(sk_company_id VARCHAR(64), nome_produtora VARCHAR(255))
- dim_genres(sk_genre_id VARCHAR(64), nome_genero VARCHAR(50))
- dim_movies(sk_movie_id VARCHAR(64), id_filme VARCHAR(50), titulo VARCHAR(500), data_lancamento DATE, ano_lancamento INTEGER, duracao_minutos INTEGER, 